In [1]:
# ============================================================
# FINAL SUBMISSION — CAR MODEL DETECTION EVIDENCE PIPELINE
# YOLOv8 Vehicle Detection + Transfer-Learning Car Model Classifier
# One-cell Kaggle-ready code: training, inference, evidence, metrics
# ============================================================

import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")  # Set "" to force CPU
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import sys
import subprocess

REQUIRED_PACKAGES = [
    "ultralytics",
    "opencv-python-headless",
    "numpy",
    "pandas",
    "tqdm",
    "pillow",
    "scikit-learn",
    "torch",
    "torchvision",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", *REQUIRED_PACKAGES, "--quiet"],
    check=False,
)

import csv
import json
import math
import random
import shutil
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms
from tqdm import tqdm
from ultralytics import YOLO

# ============================================================
# CONFIGURATION — EDIT THESE PATHS FOR YOUR KAGGLE DATASETS
# ============================================================

# Kaggle datasets requested for this task. Add these five datasets to your
# notebook through Kaggle's "Add Data" panel, then run this cell as-is.
#
# Classification datasets used to train make/model labels:
#   https://www.kaggle.com/datasets/jutrera/stanford-car-dataset-by-classes-folder
#   https://www.kaggle.com/datasets/abhishektyagi001/vehicle-make-model-recognition-dataset-vmmrdb
#
# Detection/video datasets used for evidence/inference output:
#   https://www.kaggle.com/datasets/seyeon040768/car-detection-dataset
#   https://www.kaggle.com/datasets/pratikbarua/vehicle-detection-dataset
#   https://www.kaggle.com/datasets/amitkumargurjar/car-detection-and-tracking-dataset
CLASSIFICATION_DATASETS = [
    "/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder",
    "/kaggle/input/datasets/abhishektyagi001/vehicle-make-model-recognition-dataset-vmmrdb"
]

DETECTION_INFERENCE_DATASETS = [
    "/kaggle/input/datasets/seyeon040768/car-detection-dataset",
    "/kaggle/input/datasets/pratikbarua/vehicle-detection-dataset",
    "/kaggle/input/datasets/amitkumargurjar/car-detection-and-tracking-dataset",
]

DATASETS = CLASSIFICATION_DATASETS + DETECTION_INFERENCE_DATASETS

# Kaggle normally mounts a dataset at /kaggle/input/<dataset-slug>,
# but the visible folder can vary when a dataset is copied, renamed, or
# versioned. These aliases let the code find the datasets even if the exact
# folder name differs from the URL slug.
DATASET_FALLBACK_ALIASES = {
    "/kaggle/input/stanford-car-dataset-by-classes-folder": [
        "stanford-car-dataset-by-classes-folder",
        "stanford-cars-dataset",
        "stanford-cars",
        "cars196",
    ],
    "/kaggle/input/vehicle-make-model-recognition-dataset-vmmrdb": [
        "vehicle-make-model-recognition-dataset-vmmrdb",
        "vmmrdb",
        "vehicle-make-model-recognition",
        "vehicle-make-model",
    ],
    "/kaggle/input/car-detection-dataset": [
        "car-detection-dataset",
        "car-detection",
    ],
    "/kaggle/input/vehicle-detection-dataset": [
        "vehicle-detection-dataset",
        "vehicle-detection",
    ],
    "/kaggle/input/car-detection-and-tracking-dataset": [
        "car-detection-and-tracking-dataset",
        "car-detection-tracking",
        "car-tracking-dataset",
    ],
}

OUTPUT_DIR = Path("/kaggle/working/car_model_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# ============================================================
# MODEL / TRAINING PARAMETERS
# ============================================================

# YOLO COCO vehicle classes: 2=car, 3=motorcycle, 5=bus, 7=truck
VEHICLE_CLASSES = {2, 5, 7}
YOLO_WEIGHTS = "yolov8n.pt"       # use yolov8s.pt/yolov8m.pt for better accuracy if runtime allows
YOLO_CONF = 0.35
YOLO_IOU = 0.50
MIN_VEHICLE_AREA_RATIO = 0.015     # ignore tiny far-away vehicles

IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 8                         # raise to 15-25 for stronger final accuracy
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
MIN_IMAGES_PER_CLASS = 8           # low classes are noisy and hurt model accuracy
MAX_CLASSES = 196                  # keep Stanford Cars scale; set None for all classes
VAL_SIZE = 0.20
NUM_WORKERS = 2

# Evidence throttling for videos: save at most one event per track/class every N frames.
VIDEO_SAMPLE_EVERY_N_FRAMES = 5
MAX_EVIDENCE_PER_VIDEO = 80
TOPK = 3

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

# ============================================================
# DATA DISCOVERY HELPERS
# ============================================================

def now_utc_iso() -> str:
    return datetime.now(UTC).isoformat().replace("+00:00", "Z")


def clean_label(label: str) -> str:
    label = str(label).strip().replace("/", "_").replace("\\", "_")
    label = "_".join(label.replace("-", " ").split())
    return label or "unknown"


def safe_stem(name: str, limit: int = 45) -> str:
    stem = Path(name).stem
    keep = "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in stem)
    return keep[:limit] or "media"


def normalize_name(name: str) -> str:
    return "".join(ch.lower() for ch in str(name) if ch.isalnum())


def available_kaggle_inputs() -> List[Path]:
    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.exists():
        return []
    return sorted([p for p in kaggle_input.iterdir() if p.is_dir()])


def resolve_dataset_path(requested_path: str) -> Optional[Path]:
    """Resolve a Kaggle dataset path using exact, alias, and fuzzy matches."""
    requested = Path(requested_path)
    if requested.exists():
        return requested

    input_dirs = available_kaggle_inputs()
    aliases = DATASET_FALLBACK_ALIASES.get(str(requested), []) + [requested.name]

    # 1) Direct alias folder check under /kaggle/input.
    for alias in aliases:
        candidate = Path("/kaggle/input") / alias
        if candidate.exists():
            return candidate

    # 2) Fuzzy alias check against available Kaggle input folder names.
    alias_keys = [normalize_name(alias) for alias in aliases]
    for folder in input_dirs:
        folder_key = normalize_name(folder.name)
        if any(alias_key and (alias_key in folder_key or folder_key in alias_key) for alias_key in alias_keys):
            return folder

    return None


def existing_dataset_roots(paths: Sequence[str], purpose: str = "dataset", allow_all_fallback: bool = False) -> List[Path]:
    roots = []
    for p in paths:
        resolved = resolve_dataset_path(p)
        if resolved is not None:
            if resolved not in roots:
                roots.append(resolved)
                if str(resolved) != str(Path(p)):
                    print(f"Resolved {purpose}: {p} -> {resolved}")
        else:
            print(f"Dataset path not found for {purpose}, skipping: {p}")

    if not roots and allow_all_fallback:
        roots = available_kaggle_inputs()
        if roots:
            print("WARNING: Requested dataset paths were not found. Falling back to all folders under /kaggle/input:")
            for folder in roots:
                print(f"  - {folder}")

    if not roots:
        available = available_kaggle_inputs()
        if available:
            print("Available /kaggle/input folders:")
            for folder in available:
                print(f"  - {folder.name}")
        else:
            print("No /kaggle/input folders are visible. In Kaggle, click '+ Add Data' and attach the requested datasets.")

    return roots


def find_media_files(roots: Sequence[Path]) -> Tuple[List[Path], List[Path]]:
    image_files, video_files = [], []
    for root in roots:
        for path in root.rglob("*"):
            if not path.is_file():
                continue
            suffix = path.suffix.lower()
            if suffix in IMAGE_EXTENSIONS:
                image_files.append(path)
            elif suffix in VIDEO_EXTENSIONS:
                video_files.append(path)
    return sorted(set(image_files)), sorted(set(video_files))


def discover_csv_labels(roots: Sequence[Path]) -> List[Tuple[Path, str]]:
    """Return (image_path, label) pairs from common Kaggle CSV annotation styles."""
    pairs = []
    path_columns = ["image", "image_path", "path", "filename", "file", "fname", "img", "relative_path"]
    make_columns = ["make", "brand", "manufacturer"]
    model_columns = ["model", "car_model", "vehicle_model", "class", "label", "name"]

    for root in roots:
        for csv_path in root.rglob("*.csv"):
            try:
                df = pd.read_csv(csv_path)
            except Exception as exc:
                print(f"Could not read CSV {csv_path}: {exc}")
                continue

            lower_to_original = {c.lower().strip(): c for c in df.columns}
            img_col = next((lower_to_original[c] for c in path_columns if c in lower_to_original), None)
            if img_col is None:
                continue

            make_col = next((lower_to_original[c] for c in make_columns if c in lower_to_original), None)
            model_col = next((lower_to_original[c] for c in model_columns if c in lower_to_original), None)
            if model_col is None:
                continue

            for _, row in df.iterrows():
                raw_path = str(row[img_col])
                candidates = [root / raw_path, csv_path.parent / raw_path, root / Path(raw_path).name]
                image_path = next((c for c in candidates if c.exists()), None)
                if image_path is None:
                    matches = list(root.rglob(Path(raw_path).name))
                    image_path = matches[0] if matches else None
                if image_path is None or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                    continue

                parts = []
                if make_col is not None and not pd.isna(row[make_col]):
                    parts.append(str(row[make_col]))
                if model_col is not None and not pd.isna(row[model_col]):
                    parts.append(str(row[model_col]))
                label = clean_label(" ".join(parts))
                if label.lower() != "unknown":
                    pairs.append((image_path, label))

    return pairs


def discover_folder_labels(image_files: Sequence[Path], roots: Sequence[Path]) -> List[Tuple[Path, str]]:
    """Infer labels from class folders, preferring train/valid/test child folder names."""
    pairs = []
    split_names = {"train", "training", "valid", "val", "validation", "test", "testing"}
    generic_names = {"images", "image", "imgs", "data", "dataset", "archive", "cars", "car", "vehicles", "vehicle"}

    for img in image_files:
        rel_parts = None
        for root in roots:
            try:
                rel_parts = img.relative_to(root).parts
                break
            except ValueError:
                continue
        if not rel_parts or len(rel_parts) < 2:
            continue

        parents = list(img.parent.parts)
        label = None
        for idx, part in enumerate(parents):
            if part.lower() in split_names and idx + 1 < len(parents):
                label = parents[idx + 1]
                break
        if label is None:
            candidate = img.parent.name
            if candidate.lower() in generic_names and len(parents) >= 2:
                candidate = parents[-2]
            label = candidate

        label = clean_label(label)
        if label.lower() not in generic_names and label.lower() not in split_names:
            pairs.append((img, label))
    return pairs


def build_labeled_dataset(roots: Sequence[Path]) -> List[Tuple[Path, str]]:
    """Build classifier labels only from make/model classification datasets.

    Detection datasets often contain folders named "train/images" or YOLO label
    files, which are not car model labels. Keeping classification roots separate
    prevents accidental labels such as "images", "valid", or "dataset" from
    entering the classifier.
    """
    image_files, _ = find_media_files(roots)
    csv_pairs = discover_csv_labels(roots)
    folder_pairs = discover_folder_labels(image_files, roots)

    # CSV labels are usually more precise; use them when available, otherwise folder labels.
    pairs = csv_pairs if len(csv_pairs) >= 100 else folder_pairs

    # Deduplicate and filter weak labels.
    dedup = {}
    for path, label in pairs:
        dedup[str(path)] = clean_label(label)
    pairs = [(Path(p), y) for p, y in dedup.items()]

    counts = Counter(y for _, y in pairs)
    labels = [y for y, n in counts.most_common(MAX_CLASSES) if n >= MIN_IMAGES_PER_CLASS]
    allowed = set(labels)
    pairs = [(p, y) for p, y in pairs if y in allowed]

    print(f"Discovered {len(image_files)} classification images.")
    print(f"Labeled training candidates after filtering: {len(pairs)} images across {len(allowed)} classes.")
    if allowed:
        print("Top classes:")
        for label, count in Counter(y for _, y in pairs).most_common(10):
            print(f"  {label}: {count}")

    return pairs

# ============================================================
# DATASET AND CLASSIFIER TRAINING
# ============================================================

class CarImageDataset(Dataset):
    def __init__(self, samples: Sequence[Tuple[Path, int]], transform):
        self.samples = list(samples)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, label_idx = self.samples[idx]
        try:
            image = Image.open(path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (IMAGE_SIZE, IMAGE_SIZE), (0, 0, 0))
        return self.transform(image), label_idx


def make_transforms() -> Tuple[transforms.Compose, transforms.Compose]:
    train_tfms = transforms.Compose([
        transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.72, 1.0), ratio=(0.85, 1.15)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.15, hue=0.03),
        transforms.RandomRotation(degrees=4),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.15),
    ])
    val_tfms = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return train_tfms, val_tfms


def create_classifier(num_classes: int) -> nn.Module:
    weights = models.EfficientNet_B0_Weights.DEFAULT
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.35),
        nn.Linear(in_features, num_classes),
    )
    return model


def train_classifier(labeled_pairs: Sequence[Tuple[Path, str]]):
    if len(labeled_pairs) < 2:
        print("No usable labeled images found. The pipeline will still detect vehicles, but model names will be 'unknown'.")
        return None, [], {}, None

    labels = sorted({label for _, label in labeled_pairs})
    label_to_idx = {label: idx for idx, label in enumerate(labels)}
    encoded = [(path, label_to_idx[label]) for path, label in labeled_pairs]

    y = [idx for _, idx in encoded]
    stratify = y if min(Counter(y).values()) >= 2 else None
    train_samples, val_samples = train_test_split(
        encoded,
        test_size=VAL_SIZE,
        random_state=RANDOM_SEED,
        stratify=stratify,
    )

    train_tfms, val_tfms = make_transforms()
    train_ds = CarImageDataset(train_samples, train_tfms)
    val_ds = CarImageDataset(val_samples, val_tfms)

    train_counts = Counter(label for _, label in train_samples)
    sample_weights = [1.0 / train_counts[label] for _, label in train_samples]
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))

    model = create_classifier(len(labels)).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.08)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(EPOCHS, 1))
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    best_acc = -1.0
    best_path = OUTPUT_DIR / "best_car_model_classifier.pth"
    history = []

    print("\n" + "=" * 60)
    print("TRAINING CAR MODEL CLASSIFIER")
    print("=" * 60)
    print(f"Classes: {len(labels)} | train: {len(train_ds)} | val: {len(val_ds)}")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, targets in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} train", leave=False):
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                logits = model(images)
                loss = criterion(logits, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * targets.size(0)
            train_correct += (logits.argmax(1) == targets).sum().item()
            train_total += targets.size(0)

        scheduler.step()
        train_loss /= max(train_total, 1)
        train_acc = train_correct / max(train_total, 1)

        val_metrics = evaluate_classifier(model, val_loader, labels)
        val_acc = val_metrics["accuracy"]
        history.append({
            "epoch": epoch,
            "train_loss": round(train_loss, 5),
            "train_accuracy": round(train_acc, 5),
            **{k: round(v, 5) if isinstance(v, float) else v for k, v in val_metrics.items() if k != "classification_report"},
        })

        print(
            f"Epoch {epoch:02d}: train_loss={train_loss:.4f} "
            f"train_acc={train_acc*100:.2f}% val_acc={val_acc*100:.2f}% "
            f"val_macro_f1={val_metrics['macro_f1']*100:.2f}%"
        )

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save({
                "model_state_dict": model.state_dict(),
                "labels": labels,
                "image_size": IMAGE_SIZE,
                "val_accuracy": val_acc,
                "timestamp_utc": now_utc_iso(),
            }, best_path)

    checkpoint = torch.load(best_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    final_metrics = evaluate_classifier(model, val_loader, labels)
    with open(OUTPUT_DIR / "classifier_metrics.json", "w") as f:
        json.dump({"history": history, "final": final_metrics, "labels": labels}, f, indent=2)

    print("\nClassifier validation results:")
    print(f"  Accuracy : {final_metrics['accuracy'] * 100:.2f}%")
    print(f"  Precision: {final_metrics['macro_precision'] * 100:.2f}% macro")
    print(f"  Recall   : {final_metrics['macro_recall'] * 100:.2f}% macro")
    print(f"  F1 Score : {final_metrics['macro_f1'] * 100:.2f}% macro")
    print(f"  Saved    : {best_path}")

    idx_to_label = {idx: label for label, idx in label_to_idx.items()}
    return model, labels, idx_to_label, val_tfms


@torch.no_grad()
def evaluate_classifier(model: nn.Module, loader: DataLoader, labels: Sequence[str]) -> Dict:
    model.eval()
    all_true, all_pred = [], []
    for images, targets in tqdm(loader, desc="validate", leave=False):
        images = images.to(DEVICE, non_blocking=True)
        logits = model(images)
        preds = logits.argmax(1).cpu().numpy().tolist()
        all_pred.extend(preds)
        all_true.extend(targets.numpy().tolist())

    if not all_true:
        return {"accuracy": 0.0, "macro_precision": 0.0, "macro_recall": 0.0, "macro_f1": 0.0, "classification_report": {}}

    present = sorted(set(all_true) | set(all_pred))
    present_names = [labels[i] for i in present]
    return {
        "accuracy": float(accuracy_score(all_true, all_pred)),
        "macro_precision": float(precision_score(all_true, all_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(all_true, all_pred, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(all_true, all_pred, average="macro", zero_division=0)),
        "confusion_matrix": confusion_matrix(all_true, all_pred).tolist(),
        "classification_report": classification_report(all_true, all_pred, labels=present, target_names=present_names, zero_division=0, output_dict=True),
    }

# ============================================================
# DETECTION / CLASSIFICATION HELPERS
# ============================================================

@dataclass
class DetectionResult:
    label: str
    confidence: float
    topk: List[Dict]
    bbox_ltrb: List[int]
    detector_confidence: float
    vehicle_class: str


def crop_ltrb(image_bgr: np.ndarray, bbox: Sequence[int], pad_ratio: float = 0.08) -> np.ndarray:
    h, w = image_bgr.shape[:2]
    l, t, r, b = map(int, bbox)
    bw, bh = r - l, b - t
    pad_x, pad_y = int(bw * pad_ratio), int(bh * pad_ratio)
    l, t = max(0, l - pad_x), max(0, t - pad_y)
    r, b = min(w - 1, r + pad_x), min(h - 1, b + pad_y)
    return image_bgr[t:b, l:r].copy()


@torch.no_grad()
def classify_crop(crop_bgr: np.ndarray, classifier, labels: Sequence[str], transform) -> Tuple[str, float, List[Dict]]:
    if classifier is None or transform is None or not labels or crop_bgr.size == 0:
        return "unknown", 0.0, []
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(crop_rgb)
    tensor = transform(pil).unsqueeze(0).to(DEVICE)
    logits = classifier(tensor)
    probs = torch.softmax(logits, dim=1)[0]
    k = min(TOPK, len(labels))
    values, indices = torch.topk(probs, k=k)
    topk = [
        {"label": labels[int(idx)], "confidence": round(float(val), 5)}
        for val, idx in zip(values.cpu(), indices.cpu())
    ]
    return topk[0]["label"], float(topk[0]["confidence"]), topk


def detect_vehicles(frame_bgr: np.ndarray, yolo_model: YOLO) -> List[Tuple[List[int], float, str]]:
    h, w = frame_bgr.shape[:2]
    min_area = h * w * MIN_VEHICLE_AREA_RATIO
    result = yolo_model.predict(frame_bgr, conf=YOLO_CONF, iou=YOLO_IOU, device=DEVICE, verbose=False)[0]
    names = result.names
    detections = []
    for box in result.boxes:
        cls = int(box.cls.item())
        conf = float(box.conf.item())
        if cls not in VEHICLE_CLASSES:
            continue
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int).tolist()
        area = max(0, x2 - x1) * max(0, y2 - y1)
        if area < min_area:
            continue
        detections.append(([x1, y1, x2, y2], conf, str(names.get(cls, cls))))
    return detections


def draw_detection(frame: np.ndarray, detection: DetectionResult) -> None:
    l, t, r, b = detection.bbox_ltrb
    color = (0, 180, 255) if detection.label == "unknown" else (0, 255, 0)
    cv2.rectangle(frame, (l, t), (r, b), color, 2)
    text = f"{detection.label} {detection.confidence*100:.1f}%"
    cv2.putText(frame, text, (l, max(20, t - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

# ============================================================
# EVIDENCE WRITERS
# ============================================================

event_counter = [0]

def make_event_dir(media_name: str, frame_id: Optional[int], label: str) -> Tuple[Path, str]:
    event_counter[0] += 1
    eid = f"event_{event_counter[0]:05d}"
    frame_part = f"_frame{frame_id}" if frame_id is not None else ""
    folder = OUTPUT_DIR / f"{eid}_{clean_label(label)}_{safe_stem(media_name)}{frame_part}"
    folder.mkdir(parents=True, exist_ok=True)
    return folder, eid


def save_evidence(event_dir: Path, event_id: str, source_path: Path, detection: DetectionResult, crop: np.ndarray,
                  annotated_frame: np.ndarray, frame_id: Optional[int] = None, time_sec: Optional[float] = None) -> Dict:
    crop_path = event_dir / "vehicle_crop.jpg"
    frame_path = event_dir / "annotated_frame.jpg"
    cv2.imwrite(str(crop_path), crop)
    cv2.imwrite(str(frame_path), annotated_frame)

    evidence = {
        "event_id": event_id,
        "source_media": source_path.name,
        "source_path": str(source_path),
        "frame_number": frame_id,
        "time_seconds": None if time_sec is None else round(float(time_sec), 3),
        "timestamp_utc": now_utc_iso(),
        "predicted_car_model": detection.label,
        "classification_confidence": round(detection.confidence, 5),
        "topk_predictions": detection.topk,
        "bbox_ltrb": detection.bbox_ltrb,
        "detector_confidence": round(detection.detector_confidence, 5),
        "vehicle_class": detection.vehicle_class,
        "thresholds_used": {
            "YOLO_CONF": YOLO_CONF,
            "YOLO_IOU": YOLO_IOU,
            "MIN_VEHICLE_AREA_RATIO": MIN_VEHICLE_AREA_RATIO,
            "TOPK": TOPK,
        },
        "files": {
            "vehicle_crop": crop_path.name,
            "annotated_frame": frame_path.name,
        },
    }
    with open(event_dir / "evidence.json", "w") as f:
        json.dump(evidence, f, indent=2)
    return evidence

# ============================================================
# IMAGE AND VIDEO PROCESSING
# ============================================================

def process_image(image_path: Path, yolo_model: YOLO, classifier, labels: Sequence[str], transform, csv_writer) -> int:
    frame = cv2.imread(str(image_path))
    if frame is None:
        return 0
    annotated = frame.copy()
    count = 0

    for bbox, det_conf, vehicle_class in detect_vehicles(frame, yolo_model):
        crop = crop_ltrb(frame, bbox)
        label, cls_conf, topk = classify_crop(crop, classifier, labels, transform)
        detection = DetectionResult(label, cls_conf, topk, bbox, det_conf, vehicle_class)
        draw_detection(annotated, detection)

        event_dir, eid = make_event_dir(image_path.name, None, label)
        save_evidence(event_dir, eid, image_path, detection, crop, annotated.copy())
        csv_writer.writerow([eid, image_path.name, "image", "", "", *bbox, vehicle_class, det_conf, label, cls_conf, json.dumps(topk)])
        count += 1

    if count > 0:
        cv2.imwrite(str(OUTPUT_DIR / f"OUT_{safe_stem(image_path.name)}.jpg"), annotated)
    return count


def process_video(video_path: Path, yolo_model: YOLO, classifier, labels: Sequence[str], transform, csv_writer) -> Tuple[int, int]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"  SKIP video cannot open: {video_path}")
        return 0, 0

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out_path = OUTPUT_DIR / f"OUT_{safe_stem(video_path.name)}.mp4"
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (frame_w, frame_h))

    frame_id = 0
    evidence_count = 0
    processed_frames = 0

    pbar = tqdm(total=total_frames if total_frames > 0 else None, desc=video_path.name, leave=False)
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        frame_id += 1
        pbar.update(1)

        annotated = frame.copy()
        if frame_id % VIDEO_SAMPLE_EVERY_N_FRAMES == 0:
            detections = detect_vehicles(frame, yolo_model)
            processed_frames += 1
            for bbox, det_conf, vehicle_class in detections:
                crop = crop_ltrb(frame, bbox)
                label, cls_conf, topk = classify_crop(crop, classifier, labels, transform)
                detection = DetectionResult(label, cls_conf, topk, bbox, det_conf, vehicle_class)
                draw_detection(annotated, detection)

                if evidence_count < MAX_EVIDENCE_PER_VIDEO:
                    event_dir, eid = make_event_dir(video_path.name, frame_id, label)
                    evidence = save_evidence(
                        event_dir,
                        eid,
                        video_path,
                        detection,
                        crop,
                        annotated.copy(),
                        frame_id=frame_id,
                        time_sec=frame_id / fps,
                    )
                    csv_writer.writerow([eid, video_path.name, "video", frame_id, evidence["time_seconds"], *bbox, vehicle_class, det_conf, label, cls_conf, json.dumps(topk)])
                    evidence_count += 1

        writer.write(annotated)

    pbar.close()
    cap.release()
    writer.release()
    print(f"  Done: {video_path.name} | sampled_frames={processed_frames} | evidence={evidence_count}")
    return evidence_count, frame_id

# ============================================================
# MAIN EXECUTION
# ============================================================

roots = existing_dataset_roots(DATASETS, purpose="requested dataset", allow_all_fallback=True)
classification_roots = existing_dataset_roots(CLASSIFICATION_DATASETS, purpose="classification dataset")
if not roots:
    raise FileNotFoundError(
        "No Kaggle input datasets are visible. In the notebook right sidebar, click '+ Add Data' "
        "and attach the five requested datasets, then restart/run the cell."
    )
if not classification_roots:
    print(
        "WARNING: Stanford Cars/VMMRdb classification folders were not found. "
        "The code will try to infer labels from all visible inputs, but best car-model accuracy "
        "requires adding the two classification datasets."
    )
    classification_roots = roots

labeled_pairs = build_labeled_dataset(classification_roots)
all_images, all_videos = find_media_files(roots)
print(f"Total media available for inference/evidence: {len(all_images)} images and {len(all_videos)} videos.")
classifier, class_names, idx_to_label, inference_transform = train_classifier(labeled_pairs)

print("\n" + "=" * 60)
print("LOADING VEHICLE DETECTOR")
print("=" * 60)
yolo_model = YOLO(YOLO_WEIGHTS)

# Inference set: if labeled images exist, run on a manageable sample for evidence; always include videos.
unlabeled_or_sample_images = [p for p in all_images if p not in {x for x, _ in labeled_pairs}]
if not unlabeled_or_sample_images:
    unlabeled_or_sample_images = [p for p, _ in random.sample(labeled_pairs, min(100, len(labeled_pairs)))] if labeled_pairs else all_images[:100]
else:
    unlabeled_or_sample_images = random.sample(unlabeled_or_sample_images, min(200, len(unlabeled_or_sample_images)))

summary = {
    "run_timestamp": now_utc_iso(),
    "device": DEVICE,
    "dataset_roots": [str(r) for r in roots],
    "classification_dataset_roots": [str(r) for r in classification_roots],
    "detection_inference_dataset_roots": [str(r) for r in existing_dataset_roots(DETECTION_INFERENCE_DATASETS, purpose="detection/inference dataset")],
    "num_training_images": len(labeled_pairs),
    "num_classes": len(class_names),
    "classes": class_names,
    "num_inference_images": len(unlabeled_or_sample_images),
    "num_inference_videos": len(all_videos),
    "total_events": 0,
    "total_video_frames": 0,
    "outputs": str(OUTPUT_DIR),
}

csv_path = OUTPUT_DIR / "car_model_detections.csv"
with open(csv_path, "w", newline="") as f:
    writer_csv = csv.writer(f)
    writer_csv.writerow([
        "event_id", "source_media", "media_type", "frame", "time_sec",
        "bbox_l", "bbox_t", "bbox_r", "bbox_b",
        "vehicle_class", "detector_confidence", "predicted_car_model", "classification_confidence", "topk_json",
    ])

    print("\n" + "=" * 60)
    print("PROCESSING IMAGES")
    print("=" * 60)
    for image_path in tqdm(unlabeled_or_sample_images, desc="images"):
        summary["total_events"] += process_image(image_path, yolo_model, classifier, class_names, inference_transform, writer_csv)

    print("\n" + "=" * 60)
    print("PROCESSING VIDEOS")
    print("=" * 60)
    for video_path in tqdm(all_videos, desc="videos"):
        events, frames = process_video(video_path, yolo_model, classifier, class_names, inference_transform, writer_csv)
        summary["total_events"] += events
        summary["total_video_frames"] += frames

with open(OUTPUT_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Save final model artifact if a classifier was trained.
if classifier is not None:
    torch.save({
        "model_state_dict": classifier.state_dict(),
        "labels": class_names,
        "image_size": IMAGE_SIZE,
        "architecture": "efficientnet_b0",
        "timestamp_utc": now_utc_iso(),
    }, OUTPUT_DIR / "car_model_classifier_final.pth")

print("\n" + "=" * 60)
print("FINAL RESULTS")
print("=" * 60)
print(f"  Classes learned          : {summary['num_classes']}")
print(f"  Training images used     : {summary['num_training_images']}")
print(f"  Inference images sampled : {summary['num_inference_images']}")
print(f"  Inference videos found   : {summary['num_inference_videos']}")
print(f"  Vehicle/model events     : {summary['total_events']}")
print(f"  Video frames read        : {summary['total_video_frames']}")
print(f"\nAll outputs saved to: {OUTPUT_DIR}/")
print("  Per detection evidence folder contains:")
print("    vehicle_crop.jpg, annotated_frame.jpg, evidence.json")
print("  Global files:")
print("    car_model_detections.csv, summary.json, classifier_metrics.json, car_model_classifier_final.pth")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Using device: cuda
Discovered 28021 classification images.
Labeled training candidates after filtering: 25525 images across 196 classes.
Top classes:
  chevrolet_silverado_2004: 1302
  honda_civic_1998: 1255
  chevrolet_impala_2008: 1221
  ford_f150_2006: 1197
  honda_accord_1997: 1002
  dodge_ram_2001: 499
  gmc_sierra_2012: 427
  nissan_altima_2014: 394
  toyota_camry_2014: 390
  chevrolet_impala_2007: 386
Total media available for inference/evidence: 48208 images and 0 videos.
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints

100%|██████████| 20.5M/20.5M [00:00<00:00, 153MB/s]
/tmp/ipykernel_23/2910501673.py:456: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))



TRAINING CAR MODEL CLASSIFIER
Classes: 196 | train: 20420 | val: 5105


Epoch 1/8 train:   0%|          | 0/639 [00:00<?, ?it/s]/tmp/ipykernel_23/2910501673.py:478: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):


Epoch 01: train_loss=3.3142 train_acc=35.80% val_acc=52.14% val_macro_f1=62.34%


...


Classifier validation results:
  Accuracy : 70.83%
  Precision: 84.70% macro
  Recall   : 84.70% macro
  F1 Score : 83.86% macro
  Saved    : /kaggle/working/car_model_output/best_car_model_classifier.pth

LOADING VEHICLE DETECTOR

PROCESSING IMAGES


images: 100%|██████████| 200/200 [00:11<00:00, 17.73it/s]



PROCESSING VIDEOS


videos: 0it [00:00, ?it/s]



FINAL RESULTS
  Classes learned          : 196
  Training images used     : 25525
  Inference images sampled : 200
  Inference videos found   : 0
  Vehicle/model events     : 273
  Video frames read        : 0

All outputs saved to: /kaggle/working/car_model_output/
  Per detection evidence folder contains:
    vehicle_crop.jpg, annotated_frame.jpg, evidence.json
  Global files:
    car_model_detections.csv, summary.json, classifier_metrics.json, car_model_classifier_final.pth
